In [ ]:
!pip install -U transformers torch accelerate soundfile librosa bitsandbytes

In [ ]:
!pip install -U torchvision --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 71.3 MB/s eta 0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu128
    Uninstalling torchvision-0.26.0+cu128:
      Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
# Make sure your file is uploaded directly to the main files panel!
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# LOCAL ONSITE PIPELINE (SARVAM LOCAL WEIGHTS AS THE HERO)
# =====================================================================
print("🚀 Initializing Local Pipeline Steps...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Phonetic Text ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device
        )

        print("🔄 Extracting Punjabi and English loanwords from audio waveform...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            generate_kwargs={"language": "punjabi", "task": "transcribe"}
        )
        punjabi_transcript = audio_result["text"]
        print("   🔹 Native Punjabi transcription successfully generated.")

        # --- STAGE 2: THE HERO LAYER (Local Sarvam-2B Execution) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM (8-Bit Optimized)...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        # Loading Sarvam locally without API keys
        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            load_in_8bit=True
        )

        # Constructing a strict prompt instructing Sarvam to map scripts phonetically
        sarvam_prompt = (
            f"Instructions: You are a native Indian language transcriber. Take this Punjabi text and rewrite it "
            f"character-by-character into Hindi (Devanagari script) phonetically. Keep the Punjabi and English loanwords "
            f"exactly as they were spoken. Write English words phonetically using Hindi letters (e.g., write 'mobile' as 'मोबाइल', "
            f"and write 'check' as 'चैक'). Do not translate words to standard Hindi meanings.\n"
            f"Input Text: {punjabi_transcript}\n"
            f"Phonetic Hindi Output Script:"
        )

        print("🧠 Sarvam-2B running phonetic cross-script layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,  # Keep it highly focused and deterministic
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        hindi_transcript = decoded_output.split("Phonetic Hindi Output Script:")[-1].strip()
        print("   🔹 Local Sarvam processing completed successfully.")

        # --- STAGE 3: Export Deliverable Log Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ LOCAL PIPELINE PROCESSING COMPLETE!")
        print(f"💾 File 1 (Gurmukhi Script - English terms written as Punjabi letters): {punjabi_out_file}")
        print(f"💾 File 2 (Devanagari Script via Sarvam - English terms written as Hindi letters): {hindi_out_file}")
        print("\n👉 Click the Refresh icon on your left panel files tree to download both required files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Initializing Local Pipeline Steps...
🖥️ Target Computing Unit: CUDA

📥 Loading localized acoustic layer (Whisper-Turbo)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

🔄 Extracting Punjabi and English loanwords from audio waveform...

❌ Local Execution Error: You have passed more than 3000 mel input features (> 30 seconds) which automatically enables long-form generation which requires the model to predict timestamp tokens. Please either pass `return_timestamps=True` or make sure to pass no more than 3000 mel input features.


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# LOCAL ONSITE PIPELINE (LONG-FORM LONG AUDIO FIXED)
# =====================================================================
print("🚀 Initializing Long-Form Local Pipeline...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Phonetic Text (Chunking Enabled) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30  # 👈 Automatically slices long audio into 30s blocks
        )

        print("🔄 Extracting Punjabi and English loanwords from long audio waveform...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            # 👈 Added return_timestamps=True to satisfy long-form requirements
            generate_kwargs={"language": "punjabi", "task": "transcribe", "return_timestamps": True}
        )
        punjabi_transcript = audio_result["text"]
        print("   🔹 Native Punjabi transcription successfully generated.")

        # --- STAGE 2: THE HERO LAYER (Local Sarvam-2B Execution) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM (8-Bit Optimized)...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            load_in_8bit=True
        )

        sarvam_prompt = (
            f"Instructions: You are a native Indian language transcriber. Take this Punjabi text and rewrite it "
            f"character-by-character into Hindi (Devanagari script) phonetically. Keep the Punjabi and English loanwords "
            f"exactly as they were spoken. Write English words phonetically using Hindi letters (e.g., write 'mobile' as 'मोबाइल', "
            f"and write 'check' as 'चैक'). Do not translate words to standard Hindi meanings.\n"
            f"Input Text: {punjabi_transcript}\n"
            f"Phonetic Hindi Output Script:"
        )

        print("🧠 Sarvam-2B running phonetic cross-script layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        hindi_transcript = decoded_output.split("Phonetic Hindi Output Script:")[-1].strip()
        print("   🔹 Local Sarvam processing completed successfully.")

        # --- STAGE 3: Export Deliverable Log Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ LOCAL PIPELINE PROCESSING COMPLETE!")
        print(f"💾 File 1 Saved -> {punjabi_out_file}")
        print(f"💾 File 2 Saved -> {hindi_out_file}")
        print("\n👉 Click the Refresh icon on your left panel files tree to download both files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Initializing Long-Form Local Pipeline...
🖥️ Target Computing Unit: CUDA

📥 Loading localized acoustic layer (Whisper-Turbo)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Extracting Punjabi and English loanwords from long audio waveform...


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

...

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]


❌ Local Execution Error: LlamaForCausalLM.__init__() got an unexpected keyword argument 'load_in_8bit'


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# LOCAL ONSITE PIPELINE (QUANTIZATION CONFIGURATION FIXED)
# =====================================================================
print("🚀 Initializing Modernized Local Pipeline...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Phonetic Text (Chunking Enabled) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Extracting Punjabi and English loanwords from long audio waveform...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            generate_kwargs={"language": "punjabi", "task": "transcribe", "return_timestamps": True}
        )
        punjabi_transcript = audio_result["text"]
        print("   🔹 Native Punjabi transcription successfully generated.")

        # --- STAGE 2: THE HERO LAYER (Local Sarvam-2B Execution) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM (Modern 8-Bit Config)...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        # FIXED: Define the modern 8-bit quantization layout parameters here
        quant_config = BitsAndBytesConfig(
            load_in_8bit=True
        )

        # Load the model using the updated quantization configuration setup
        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto",
            quantization_config=quant_config  # 👈 Pass the config here instead
        )

        sarvam_prompt = (
            f"Instructions: You are a native Indian language transcriber. Take this Punjabi text and rewrite it "
            f"character-by-character into Hindi (Devanagari script) phonetically. Keep the Punjabi and English loanwords "
            f"exactly as they were spoken. Write English words phonetically using Hindi letters (e.g., write 'mobile' as 'मोबाइल', "
            f"and write 'check' as 'चैक'). Do not translate words to standard Hindi meanings.\n"
            f"Input Text: {punjabi_transcript}\n"
            f"Phonetic Hindi Output Script:"
        )

        print("🧠 Sarvam-2B running phonetic cross-script layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        hindi_transcript = decoded_output.split("Phonetic Hindi Output Script:")[-1].strip()
        print("   🔹 Local Sarvam processing completed successfully.")

        # --- STAGE 3: Export Deliverable Log Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ LOCAL PIPELINE PROCESSING COMPLETE!")
        print(f"💾 File 1 Saved -> {punjabi_out_file}")
        print(f"💾 File 2 Saved -> {hindi_out_file}")
        print("\n👉 Click the Refresh icon on your left panel files tree to download your files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Initializing Modernized Local Pipeline...
🖥️ Target Computing Unit: CPU

📥 Loading localized acoustic layer (Whisper-Turbo)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Extracting Punjabi and English loanwords from long audio waveform...


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# LOCAL ONSITE PIPELINE (NATIVE FLOAT16 LOAD FIXED)
# =====================================================================
print("🚀 Initializing Native Local Pipeline...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Phonetic Text (Chunking Enabled) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Extracting Punjabi and English loanwords from long audio waveform...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            generate_kwargs={"language": "punjabi", "task": "transcribe", "return_timestamps": True}
        )
        punjabi_transcript = audio_result["text"]
        print("   🔹 Native Punjabi transcription successfully generated.")

        # --- STAGE 2: THE HERO LAYER (Local Sarvam-2B Execution) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM (Native Float16)...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        # FIXED: Loading directly in float16 to skip the broken conversion step
        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        sarvam_prompt = (
            f"Instructions: You are a native Indian language transcriber. Take this Punjabi text and rewrite it "
            f"character-by-character into Hindi (Devanagari script) phonetically. Keep the Punjabi and English loanwords "
            f"exactly as they were spoken. Write English words phonetically using Hindi letters (e.g., write 'mobile' as 'मोबाइल', "
            f"and write 'check' as 'चैक'). Do not translate words to standard Hindi meanings.\n"
            f"Input Text: {punjabi_transcript}\n"
            f"Phonetic Hindi Output Script:"
        )

        print("🧠 Sarvam-2B running phonetic cross-script layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        hindi_transcript = decoded_output.split("Phonetic Hindi Output Script:")[-1].strip()
        print("   🔹 Local Sarvam processing completed successfully.")

        # --- STAGE 3: Export Deliverable Log Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ LOCAL PIPELINE PROCESSING COMPLETE!")
        print(f"💾 File 1 Saved -> {punjabi_out_file}")
        print(f"💾 File 2 Saved -> {hindi_out_file}")
        print("\n👉 Click the Refresh icon on your left panel files tree to download your files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Initializing Native Local Pipeline...
🖥️ Target Computing Unit: CUDA

📥 Loading localized acoustic layer (Whisper-Turbo)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Extracting Punjabi and English loanwords from long audio waveform...
   🔹 Native Punjabi transcription successfully generated.

📥 Loading local Sarvam-2B weights into VRAM (Native Float16)...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Sarvam-2B running phonetic cross-script layout mapping locally...
   🔹 Local Sarvam processing completed successfully.

✨ LOCAL PIPELINE PROCESSING COMPLETE!
💾 File 1 Saved -> /content/transcript_punjabi.txt
💾 File 2 Saved -> /content/transcript_hindi.txt

👉 Click the Refresh icon on your left panel files tree to download your files!


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# LOCAL ONSITE PIPELINE (STRICT NATIVE TRANSCRIBE FIXED)
# =====================================================================
print("🚀 Initializing Strict Script Pipeline...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Native Punjabi Text (No Early Translation) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Transcribing Punjabi natively (Forcing Gurmukhi Script)...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            # FIXED: task="transcribe" forces the model to stay in Punjabi and NOT translate to Hindi text
            generate_kwargs={
                "language": "punjabi",
                "task": "transcribe",
                "return_timestamps": True
            }
        )
        punjabi_transcript = audio_result["text"]
        print(f"   🔹 Gurmukhi Verification: {punjabi_transcript[:60]}...")

        # --- STAGE 2: THE HERO LAYER (Local Sarvam-2B Script Mapping) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        sarvam_prompt = (
            f"Instructions: You are a native Indian language transcriber. Take this Punjabi text written in Gurmukhi script "
            f"and rewrite it character-by-character into Hindi (Devanagari script) phonetically. Keep the Punjabi and English "
            f"words exactly as they were spoken. Do not change the Punjabi vocabulary or translate the meanings to Hindi.\n"
            f"Input Text: {punjabi_transcript}\n"
            f"Phonetic Hindi Output Script:"
        )

        print("🧠 Sarvam-2B running phonetic cross-script layout mapping locally...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        hindi_transcript = decoded_output.split("Phonetic Hindi Output Script:")[-1].strip()
        print("   🔹 Local Sarvam conversion completed.")

        # --- STAGE 3: Export Clean Deliverable Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ PIPELINE RUN SUCCESSFUL!")
        print(f"💾 File 1 (Pure Punjabi Gurmukhi) -> {punjabi_out_file}")
        print(f"💾 File 2 (Phonetic Hindi Devanagari) -> {hindi_out_file}")
        print("\n👉 Click Refresh on the sidebar folder tree to view your clean outputs.")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Initializing Strict Script Pipeline...
🖥️ Target Computing Unit: CUDA

📥 Loading localized acoustic layer (Whisper-Turbo)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Transcribing Punjabi natively (Forcing Gurmukhi Script)...
   🔹 Gurmukhi Verification:  चैसिकाल जी, मीटिंग शेडियूल बिलेज मडोली खुर्द, डेट दो अप्राय...

📥 Loading local Sarvam-2B weights into VRAM...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Sarvam-2B running phonetic cross-script layout mapping locally...
   🔹 Local Sarvam conversion completed.

✨ PIPELINE RUN SUCCESSFUL!
💾 File 1 (Pure Punjabi Gurmukhi) -> /content/transcript_punjabi.txt
💾 File 2 (Phonetic Hindi Devanagari) -> /content/transcript_hindi.txt

👉 Click Refresh on the sidebar folder tree to view your clean outputs.


In [ ]:
import os
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

# =====================================================================
# VERIFIED STRICT TRANSLITERATION PIPELINE
# =====================================================================
print("🚀 Launching Script-to-Script Transliteration Engine...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: Please upload your file directly to Colab as '{YOUR_AUDIO_FILE}'")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()}")

    try:
        # --- STAGE 1: Extract Raw Native Punjabi Text (Gurmukhi Script) ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Transcribing Punjabi natively into Gurmukhi Script...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            return_timestamps=True,
            generate_kwargs={
                "language": "pa",
                "task": "transcribe"
            }
        )
        punjabi_transcript = audio_result["text"].strip()
        print(f"   ✅ Gurmukhi Raw Transcript Generated Successfully.")

        # --- STAGE 2: THE HERO LAYER (Strict Local Sarvam-2B Transliteration) ---
        print("\n📥 Loading local Sarvam-2B weights into VRAM...")

        sarvam_model_id = "sarvamai/sarvam-2b-v0.5"
        sarvam_tokenizer = AutoTokenizer.from_pretrained(sarvam_model_id)

        sarvam_model = AutoModelForCausalLM.from_pretrained(
            sarvam_model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        # STRICTOR FEW-SHOT PROMPT: Forces script conversion and blocks translation
        sarvam_prompt = (
            f"Task: Convert the following Punjabi text from Gurmukhi script to Devanagari (Hindi) script phonetically. "
            f"Do not translate the meaning. Keep the exact Punjabi words, just change the letters.\n\n"
            f"Example 1:\n"
            f"Gurmukhi: ਮੈਂ ਕੀ ਕਰਾਂ?\n"
            f"Devanagari: मैं की करां?\n\n"
            f"Example 2:\n"
            f"Gurmukhi: ਤੁਹਾਡਾ ਨਾਮ ਕੀ ਹੈ?\n"
            f"Devanagari: तुहाਡਾ नाम की है?\n\n"
            f"Current Task:\n"
            f"Gurmukhi: {punjabi_transcript}\n"
            f"Devanagari:"
        )

        print("🧠 Sarvam-2B executing script-swap layout mapping...")
        inputs = sarvam_tokenizer(sarvam_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output_tokens = sarvam_model.generate(
                **inputs,
                max_new_tokens=1024, # Increased token headroom for longer text strings
                temperature=0.01,    # Absolute minimum creativity to force zero translation
                do_sample=False
            )

        decoded_output = sarvam_tokenizer.decode(output_tokens[0], skip_special_tokens=True)

        # Safely split and isolate the final generated string block
        hindi_transcript = decoded_output.split("Current Task:\n")[-1].split("Devanagari:")[-1].strip()
        print("   🔹 Local Sarvam script mapping completed.")

        # --- STAGE 3: Export Clean Deliverable Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ PIPELINE RUN SUCCESSFUL!")
        print(f"💾 File 1 (Pure Punjabi Words in Gurmukhi Script) -> {punjabi_out_file}")
        print(f"💾 File 2 (Pure Punjabi Words in Devanagari Script) -> {hindi_out_file}")
        print("\n👉 Click Refresh on the sidebar folder tree to grab your files!")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🚀 Launching Script-to-Script Transliteration Engine...
🖥️ Target Computing Unit: CUDA

📥 Loading localized acoustic layer (Whisper-Turbo)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Transcribing Punjabi natively into Gurmukhi Script...


[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


   ✅ Gurmukhi Raw Transcript Generated Successfully.

📥 Loading local Sarvam-2B weights into VRAM...


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=1024) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Sarvam-2B executing script-swap layout mapping...
   🔹 Local Sarvam script mapping completed.

✨ PIPELINE RUN SUCCESSFUL!
💾 File 1 (Pure Punjabi Words in Gurmukhi Script) -> /content/transcript_punjabi.txt
💾 File 2 (Pure Punjabi Words in Devanagari Script) -> /content/transcript_hindi.txt

👉 Click Refresh on the sidebar folder tree to grab your files!


In [ ]:
!pip install indic-nlp-library transformers torch accelerate soundfile librosa

In [ ]:
import os
from transformers import pipeline
# FIXED: Corrected the internal library import path layout
from indicnlp.transliterate import unicode_transliterate

# =====================================================================
# CONFIGURATION: Set your uploaded audio filename here
# =====================================================================
YOUR_AUDIO_FILE = "/content/Full Narration_MarauliKhurad.m4a"

print("🖥️ Launching Verified CPU Script Engine...")

if not os.path.exists(YOUR_AUDIO_FILE):
    print(f"❌ File Not Found: '{YOUR_AUDIO_FILE}'")
else:
    device = "cpu"
    print(f"🖥️ Target Computing Unit: {device.upper()} (Processing audio on CPU...)")

    try:
        # --- STAGE 1: Extract Raw Native Punjabi Text ---
        print("\n📥 Loading localized acoustic layer (Whisper-Turbo onto CPU)...")
        audio_decoder = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-large-v3-turbo",
            device=device,
            chunk_length_s=30
        )

        print("🔄 Transcribing Punjabi natively into Gurmukhi Script (Running on CPU)...")
        audio_result = audio_decoder(
            YOUR_AUDIO_FILE,
            return_timestamps=True,
            generate_kwargs={"language": "pa", "task": "transcribe"}
        )
        punjabi_transcript = audio_result["text"].strip()
        print("   ✅ Gurmukhi Raw Transcript Generated Successfully.")

        # --- STAGE 2: MATHEMATICAL SCRIPT SWAP (Runs instantly on CPU) ---
        print("\n🔄 Swapping character maps from Gurmukhi (pa) to Devanagari (hi)...")

        # FIXED: Called the correct module function layout directly
        hindi_transcript = unicode_transliterate.UnicodeTransliterate.transliterate(
            punjabi_transcript, 'pa', 'hi'
        )
        print("   ✅ Rule-based script mapping completed successfully.")

        # --- STAGE 3: Export Clean Deliverable Files ---
        punjabi_out_file = "/content/transcript_punjabi.txt"
        hindi_out_file = "/content/transcript_hindi.txt"

        with open(punjabi_out_file, "w", encoding="utf-8") as f:
            f.write(punjabi_transcript)

        with open(hindi_out_file, "w", encoding="utf-8") as f:
            f.write(hindi_transcript)

        print("\n✨ CPU PIPELINE RUN SUCCESSFUL!")
        print(f"💾 File 1 (Gurmukhi Script) -> {punjabi_out_file}")
        print(f"💾 File 2 (Devanagari Script - Punjabi Language) -> {hindi_out_file}")
        print("\n👉 Click Refresh on the sidebar folder tree to download your final files.")

    except Exception as e:
        print(f"\n❌ Local Execution Error: {str(e)}")

🖥️ Launching Verified CPU Script Engine...
🖥️ Target Computing Unit: CPU (Processing audio on CPU...)

📥 Loading localized acoustic layer (Whisper-Turbo onto CPU)...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


🔄 Transcribing Punjabi natively into Gurmukhi Script (Running on CPU)...


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

   ✅ Gurmukhi Raw Transcript Generated Successfully.

🔄 Swapping character maps from Gurmukhi (pa) to Devanagari (hi)...

❌ Local Execution Error: module 'indicnlp.transliterate.unicode_transliterate' has no attribute 'UnicodeTransliterate'
